In [ ]:
import requests
import time
import json
import pandas as pd
from tqdm import tqdm

BASE_URL = "http://localhost:8002/generate"

OUTPUT_DIR = "results"

In [ ]:
configs = [
    {"k": 1, "temperature": 0.0, "max_tokens": 64},
    {"k": 4, "temperature": 0.0, "max_tokens": 64},
    {"k": 8, "temperature": 0.0, "max_tokens": 64},
    
    # optional experiments
    {"k": 4, "temperature": 0.2, "max_tokens": 64},
    {"k": 4, "temperature": 0.5, "max_tokens": 64},
]

In [ ]:
prompts = [
    "Explain transformer architecture in simple terms.",
    "What is speculative decoding in large language models?",
    "Describe how GPU memory affects inference latency.",
    "Explain the difference between throughput and latency.",
    "What is KV cache in LLMs?"
]

In [ ]:
def run_request(prompt, config):
    payload = {
        "prompt": prompt,
        "max_tokens": config["max_tokens"],
        "temperature": config["temperature"],
        "k": config["k"]
    }

    start = time.time()
    r = requests.post(BASE_URL, json=payload)
    end = time.time()

    if r.status_code != 200:
        return {"error": r.text}

    data = r.json()

    return {
        "prompt": prompt,
        "k": config["k"],
        "temperature": config["temperature"],
        "ttft_ms": data.get("ttft_ms"),
        "tpot_ms": data.get("tpot_ms"),
        "e2e_ms": data.get("e2e_ms"),
        "acceptance_rate": data.get("acceptance_rate"),
        "tokens_generated": data.get("tokens_generated"),
        "speedup_ratio": data.get("speedup_ratio"),
        "latency_wall": (end - start) * 1000
    }

In [ ]:
results = []

N_RUNS = 10

for config in configs:
    print(f"\nRunning config: {config}")
    
    for i in tqdm(range(N_RUNS)):
        for prompt in prompts:
            res = run_request(prompt, config)
            results.append(res)

df = pd.DataFrame(results)
df.head()

In [ ]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = os.path.join(OUTPUT_DIR, "speculative_results.csv")
json_path = os.path.join(OUTPUT_DIR, "speculative_results.json")

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2)

print("Saved to:", csv_path, json_path)

In [ ]:
summary = df.groupby(["k", "temperature"]).agg({
    "ttft_ms": ["mean", "median"],
    "tpot_ms": ["mean"],
    "acceptance_rate": ["mean"],
    "speedup_ratio": ["mean"],
    "latency_wall": ["mean"]
}).round(2)

summary

In [ ]:
for (k, temp), group in df.groupby(["k", "temperature"]):
    print(f"\nConfig: k={k}, temp={temp}")
    print(f"Acceptance: {group['acceptance_rate'].mean():.3f}")
    print(f"Speedup:    {group['speedup_ratio'].mean():.2f}x")
    print(f"TTFT:       {group['ttft_ms'].mean():.1f} ms")